# ROAD-SHIELD — train every model, in order, in one notebook

**Runtime ▸ Change runtime type ▸ T4 GPU** (or L4 / A100), then **Runtime ▸ Run all**.

| stage | what it trains | rough time on a T4 |
|---|---|---|
| 1 · main | DNIT + RDD2022 India corpus, frozen MobileNetV2 head, four fine-tuned CNNs, served classifier chosen by a fixed rule, IMU 1-D CNN vs RandomForest, benchmarks, report | 1.5–2.5 h |
| 2 · extra | U-Net defect segmenter, YOLOv8 road-damage detector (RDD2022 India) | 1–1.75 h |
| 3 · deep | DeiT-Small and LeViT-256 transformers + two CNNs as ensemble members, ensemble/cascade chosen on validation, IMU transformer; RT-DETR if `RTDETR = True` | 1.5–2.5 h (+1.5 h RT-DETR) |
| 4 · measurements | privacy-redactor recall (WIDER FACE + licence plates), classifier confidence calibration | 15–30 min |
| 5 · package | test suite, one zip with everything that changed, a table of what is now served | 10 min |

**Resumable.** With Google Drive mounted (next cells), every finished step is saved to `MyDrive/road_shield_state`.
When Colab disconnects — a free session usually ends before all of this is done — open the notebook again and
**Run all**: finished steps are skipped and training continues where it stopped.

**Nothing is served just because it was trained.** Each stage's script picks a model only by its rule on
validation data, and on your laptop `python -m mlops intake` gates every model again against what is in
production before it replaces anything (last cell).

In [ ]:
# ---- settings (change before Run all) ----
BRANCH = "audit-2026-10-03"
RUN_MAIN, RUN_EXTRA, RUN_DEEP, RUN_MEASURE = True, True, True, True
RTDETR = False            # also train RT-DETR as a detector candidate (+~1.5 h)
EPOCHS = ""               # "" = each script's default; e.g. "5" for a quick end-to-end trial run
QUICK_ARCHS = ""          # e.g. "efficientnet_b0" to fine-tune one CNN instead of four in stage 1

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU on this runtime. Runtime > Change runtime type > T4 GPU, then Run all again.")
print("GPU:", gpu.stdout.strip())

In [ ]:
# Google Drive keeps progress between sessions. Without it the run still works, but a disconnect starts over.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted - the run will not be resumable:", e)

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("SIH_PROJECT/.git"):
    !git clone -q -b {BRANCH} https://github.com/udbhav968-creator/SIH_PROJECT.git
os.chdir("/content/SIH_PROJECT")
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only origin {BRANCH}
!git log --oneline -1
os.makedirs("logs", exist_ok=True)
env = {}
if EPOCHS: env.update(EPOCHS=EPOCHS, RDD_EPOCHS=EPOCHS)
if QUICK_ARCHS: env["ARCHS"] = QUICK_ARCHS
if RTDETR: env["RTDETR"] = "1"
os.environ.update(env)
print("settings passed to the scripts:", env or "defaults")

def finished(log, marker):
    try:
        ok = marker in open(log, encoding="utf-8", errors="replace").read()
    except FileNotFoundError:
        ok = False
    print(("FINISHED: " if ok else "NOT FINISHED (run this cell again to resume): ") + log)
    return ok

## Stage 1 · main run

In [ ]:
if RUN_MAIN:
    !bash scripts/colab_train_all.sh 2>&1 | tee -a logs/colab_run.txt | grep -vE '^\s+(epoch|batch) [0-9]+ batch'
    finished("logs/colab_run.txt", "ALL_STEPS_DONE")

## Stage 2 · U-Net segmenter and YOLOv8 detector

In [ ]:
if RUN_EXTRA:
    !bash scripts/colab_train_extra.sh 2>&1 | tee -a logs/colab_extra.txt | grep -vE '^\s+(epoch|batch) [0-9]+ batch'
    finished("logs/colab_extra.txt", "EXTRA_STEPS_DONE")

## Stage 3 · transformers, ensemble cascade, IMU transformer (RT-DETR optional)

In [ ]:
if RUN_DEEP:
    !bash scripts/colab_train_deep.sh 2>&1 | tee -a logs/colab_deep.txt | grep -vE '^\s+(epoch|batch) [0-9]+ batch'
    finished("logs/colab_deep.txt", "DEEP_STEPS_DONE")

## Stage 4 · measurements (no training)

In [ ]:
if RUN_MEASURE:
    !pip -q install huggingface_hub
    !python -m scripts.measure_redactor_recall 2>&1 | tail -15
    if not os.path.isdir("datasets/_eval_rdd2022_india"):
        !python -m scripts.ingest_rdd2022_india 2>&1 | tail -3
    !python -m scripts.measure_calibration --apply 2>&1 | tail -15

## Stage 5 · tests, one zip, what is served now

In [ ]:
!python -m scripts.build_claims 2>&1 | tail -2
!python -m unittest discover -s tests -t . > logs/tests_everything.txt 2>&1; grep -E "^(Ran|OK|FAILED)" logs/tests_everything.txt
!python -m mlops.ci_check 2>&1 | tail -15

# everything that differs from the commit this run started from (works after a resume too: restored files
# count, because git compares contents, not timestamps)
REPORT = "CSET485_ROAD_SHIELD_Milestone2_Report_Audited_TrackedChanges.docx"
st = subprocess.run(["git", "status", "--porcelain", "--untracked-files=all", "--", "checkpoints", "logs", REPORT],
                    capture_output=True, text=True).stdout.splitlines()
paths = [l[3:].strip().strip('"') for l in st if l[:2].strip() != "D"]
skip = (".db", ".db-wal", ".db-shm")
keep = [p for p in paths if os.path.isfile(p) and not p.endswith(skip) and "/.done" not in p
        and "/detectors/" not in p and not (p.startswith("checkpoints/ensemble_runs/") and p.endswith(".onnx"))
        and os.path.getsize(p) < 99 * 1024 * 1024]
import zipfile
with zipfile.ZipFile("road_shield_all_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p)
print(len(keep), "files,", round(os.path.getsize("road_shield_all_outputs.zip") / 1e6, 1), "MB -> road_shield_all_outputs.zip")
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs("/content/drive/MyDrive/road_shield_state", exist_ok=True)
    !cp road_shield_all_outputs.zip /content/drive/MyDrive/road_shield_state/
    print("copy saved to MyDrive/road_shield_state/road_shield_all_outputs.zip")

import json, glob
def j(p):
    try: return json.load(open(p))
    except Exception: return {}
sel = j("checkpoints/vision_model_selection.json"); seg = j("checkpoints/segmenter_selection.json")
ens = j("checkpoints/ensemble_report.json"); imu = j("checkpoints/imu_model_selection.json")
fs = j("checkpoints/finetune_summary.json"); rdd = j("checkpoints/damage_rdd2022_india.json")
print("\nserved now")
print("  image classifier :", sel.get("served"), "| test macro-F1", (fs.get("served_test") or {}).get("macro_f1"))
print("  ensemble         :", ens.get("decision") or "not run")
print("  segmenter        :", seg.get("served"))
print("  road-damage det. :", "mAP50", (rdd.get("test") or {}).get("map50"))
print("  IMU              :", imu.get("served"), imu.get("deep_arch") or "")

In [ ]:
from google.colab import files
files.download("road_shield_all_outputs.zip")

## On the laptop (PowerShell)

```powershell
cd C:\Users\Dell\SIH_PROJECT
git pull
python -m mlops intake "$HOME\Downloads\road_shield_all_outputs.zip"
python -m scripts.build_claims
python -m unittest discover -s tests -t .
git add checkpoints logs
git commit -m "Colab training results, taken in through the model gates"
git push origin audit-2026-10-03
```

`mlops intake` registers every model in the zip, gates it against the one in production and promotes only those
that pass (a failed one is listed with the reason and production is left alone; `python -m mlops rollback <model>`
undoes a promotion). Or run `scripts\apply_colab_outputs.ps1`, which does the same steps.